# 01 — Data Exploration
Raw sensor data (vibration, current, temperature) before any cleaning or feature engineering. Goal: understand shape, quality, and per-unit degradation patterns before trusting anything downstream.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import config
import preprocessing

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (10, 4)

## Load raw data

In [ ]:
df = preprocessing.load_data()
print(df.shape)
df.head()

## Basic shape and quality checks
Row counts per unit, missing values, and obviously-wrong readings should all be checked here — before `clean_data()` silently drops anything, know what you're about to lose.

In [ ]:
print('Units:', df[config.COL_UNIT_ID].nunique())
print('\nRows per unit:')
print(df.groupby(config.COL_UNIT_ID).size().describe())
print('\nMissing values:')
print(df[config.RAW_SENSOR_COLS].isna().sum())
print('\nDuplicate (unit_id, timestamp) rows:', 
      df.duplicated(subset=[config.COL_UNIT_ID, config.COL_TIMESTAMP]).sum())

In [ ]:
df[config.RAW_SENSOR_COLS].describe()

## Sanity-check ranges
Flag rows that `clean_data()` will remove as physically impossible so you know whether that's sensor noise or a real data problem upstream.

In [ ]:
impossible = df[
    (df[config.COL_VIBRATION] < 0)
    | (df[config.COL_CURRENT] < 0)
    | (df[config.COL_TEMPERATURE] <= -50)
    | (df[config.COL_TEMPERATURE] >= 300)
]
print(f'{len(impossible)} rows ({len(impossible)/len(df):.2%}) would be dropped as impossible readings')
impossible.head()

## Per-unit sensor trajectories
This is the key exploratory plot: do vibration/current/temperature actually trend toward failure, or is the degradation signal too noisy to be useful? If trajectories look flat until a sudden jump, your RUL labeling window (`RUL_CAP` in `config.py`) may need to be narrower.

In [ ]:
sample_units = df[config.COL_UNIT_ID].unique()[:5]

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=False)
for unit in sample_units:
    g = df[df[config.COL_UNIT_ID] == unit].sort_values(config.COL_TIMESTAMP)
    t = np.arange(len(g))
    axes[0].plot(t, g[config.COL_VIBRATION], alpha=0.7, label=f'unit {unit}')
    axes[1].plot(t, g[config.COL_CURRENT], alpha=0.7, label=f'unit {unit}')
    axes[2].plot(t, g[config.COL_TEMPERATURE], alpha=0.7, label=f'unit {unit}')

axes[0].set_title('Vibration over unit lifetime'); axes[0].legend(fontsize=7)
axes[1].set_title('Current over unit lifetime')
axes[2].set_title('Temperature over unit lifetime'); axes[2].set_xlabel('cycle index')
plt.tight_layout()
plt.show()

## Sensor correlations
Informs whether cross-signal features (e.g. vibration/current rolling correlation in `feature_engineering.py`) are likely to add signal — if sensors are already highly correlated raw, engineered cross-features may be redundant.

In [ ]:
corr = df[config.RAW_SENSOR_COLS].corr()
corr

## Next step
If sensor trajectories show visible degradation trends and data quality looks acceptable, proceed to `03_feature_engineering.ipynb`. If trajectories are flat/noisy, revisit sampling rate or sensor placement before building features on top of them — no amount of feature engineering fixes a signal that isn't there.